In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import arya
import pandas as pd

In [ ]:
import ana_utils

In [ ]:
import sys
sys.path.append("../photometry")
sys.path.append("../imaging/")
import phot_utils

In [ ]:
from astropy import units as u

In [ ]:
from dataclasses import dataclass

In [ ]:
from astropy.coordinates import SkyCoord

In [ ]:
import ana_utils

In [ ]:
from astropy.table import Table

In [ ]:
import filter_utils

In [ ]:
from plot_utils import plot_mag_comparison, plot_mag_residual_hist, plot_mag_2d_residual

In [ ]:
import calc_panstarrs_offset

In [ ]:
def get_panstarrs(objname):
    panstarrs = calc_panstarrs_offset.get_panstarrs(objname)
    panstarrs.rename_columns(
        ["G_MAG", "R_MAG", "I_MAG", "G_MAG_ERR", "R_MAG_ERR", "I_MAG_ERR"],
        ["G_MAG_PS", "R_MAG_PS", "I_MAG_PS", "G_MAG_PS_ERR", "R_MAG_PS_ERR", "I_MAG_PS_ERR"],
    )
    return panstarrs

# Yasone 1

In [ ]:
cat = ana_utils.read_catalogue("yasone1", filter_bad=False, catname="allcolours", shiftname="allcolours_panstarrs_shift")

cat_forced = ana_utils.read_catalogue("yasone1", filter_bad=False, catname="allcolours_forced", shiftname="allcolours_panstarrs_shift")
cat_psf = ana_utils.read_catalogue("yasone1", filter_bad=False, catname="allcolours_forced_psf", shiftname="allcolours_panstarrs_shift")


In [ ]:
cat_sep = ana_utils.read_catalogue("yasone1", filter_bad=False, catname="julen")
cat_julen = ana_utils.read_catalogue("yasone1", filter_bad=False, catname="allcolours_julen_stack")

In [ ]:
filter_utils.calibrate_mag_col(cat_forced, "MED_MAG_APER_3")
filter_utils.calibrate_mag_col(cat_forced, "MED_MAG_APER_LB_3")

In [ ]:
filter_utils.calibrate_mag_col(cat_psf, "MED_MAG_PSF")
filter_utils.calibrate_mag_col(cat_psf, "MED_MAG_PSF_ANA")

In [ ]:
cat_lss = Table.read("../survey_data/yasone1_lss10.csv")
cat_ps = get_panstarrs("yasone1")

In [ ]:
bins = np.linspace(15, 30, 100)
plt.hist(cat["R_MAG"], bins, histtype="step")

plt.hist(cat_lss["mag_r"], bins, histtype="step")
plt.hist(cat_ps["rMeanPSFMag"], bins, histtype="step")

In [ ]:
bins = np.linspace(15, 30, 100)
plt.hist(cat["G_MAG"], bins, histtype="step")

plt.hist(cat_lss["mag_g"], bins, histtype="step")
plt.hist(cat_ps["gMeanPSFMag"], bins, histtype="step")

In [ ]:
np.sum(np.isfinite(cat_lss["mag_g"])), np.sum(np.isfinite(cat_lss["mag_r"])), np.sum(np.isfinite(cat_lss["mag_i"]))

In [ ]:
cat_joined = phot_utils.outer_join_xmatch(cat, cat_lss, ra1="ra", dec1="dec", ra2="ra", dec2="dec")



In [ ]:
plt.scatter(cat_joined["xi"], cat_joined["eta"], c=cat_joined["G_MAG"] - cat_joined["mag_g"], vmin=-1, vmax=1, cmap="RdBu", s=1)
plt.colorbar()

In [ ]:
cat_joined = phot_utils.outer_join_xmatch(cat_forced, cat_lss, ra1="ra", dec1="dec", ra2="ra", dec2="dec")


In [ ]:
plot_mag_comparison(cat_joined, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="mag_{filt}", 
                    filters=["g", "r"],
                    ref_label="osiris",
                    cmp_label="LSS"
                    
                   )

In [ ]:
plot_mag_2d_residual(cat_joined, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="mag_{filt}", 
                    filters=["g", "r"])


In [ ]:
cat_joined = phot_utils.outer_join_xmatch(cat, cat_ps, ra1="ra", dec1="dec", ra2="ra", dec2="dec")


In [ ]:
plot_mag_comparison(cat_joined, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MAG_PS", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MAG_PS_ERR", 
                    filters=["g", "r", "i"], ref_label="osiris", cmp_label="PS"
                   )

In [ ]:
plot_mag_2d_residual(cat_joined, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MAG_PS", 
                   )

In [ ]:
plot_mag_residual_hist(cat_joined, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MAG_PS", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MAG_PS_ERR", 
                    filters=["g", "r", "i"], 
                       xlim=(-10, 10),
                       sigma_scale=0
                   )

In [ ]:
cat_joined = phot_utils.outer_join_xmatch(cat, cat_ps, ra1="ra", dec1="dec", ra2="ra", dec2="dec")


### Forced Photometry

In [ ]:
plot_mag_comparison(cat_forced, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MED_MAG_APER_3", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MED_MAG_APER_3_ERR", 
                    filters=["g", "r", "i"], ref_label="osiris", cmp_label="PS"
                   )

In [ ]:
plot_mag_residual_hist(cat_forced, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MED_MAG_APER_3", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MED_MAG_APER_3_ERR", 
                    filters=["g", "r", "i"],
                       sigma_scale=2
                   )

In [ ]:
plot_mag_2d_residual(cat_forced, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MED_MAG_APER_3", 
                   )

### Forced photometry (local background)

In [ ]:
cat_forced["G_MED_MAG_APER_LB_3"]

In [ ]:
plot_mag_comparison(cat_forced, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MED_MAG_APER_LB_3", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MED_MAG_APER_LB_3_ERR", 
                    filters=["g", "r", "i"], ref_label="osiris", cmp_label="local bkg"
                   )

In [ ]:
plot_mag_2d_residual(cat_forced, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MED_MAG_APER_LB_3", 
                   )

In [ ]:
plot_mag_residual_hist(cat_forced, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MED_MAG_APER_LB_3", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MED_MAG_APER_LB_3_ERR", 
                    filters=["g", "r", "i"],
                   )

## PSF

In [ ]:
plot_mag_comparison(cat_psf, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MED_MAG_PSF", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MED_MAG_PSF_ERR", 
                    filters=["g", "r", "i"], ref_label="osiris", cmp_label="PS"
                   )

In [ ]:
plot_mag_residual_hist(cat_psf, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="{FILT}_MED_MAG_PSF", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR", mag_cmp_err_fmt="{FILT}_MED_MAG_PSF_ERR", 
                    filters=["g", "r", "i"],
                       sigma_scale=2
                   )

In [ ]:
cat_joined = phot_utils.outer_join_xmatch(cat, cat_sep, ra1="ra", dec1="dec", ra2="ra", dec2="dec")


In [ ]:
plot_mag_comparison(cat_joined, mag_ref_fmt="{FILT}_MAG_1", mag_cmp_fmt="{FILT}_MAG_2", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR_1", mag_cmp_err_fmt="{FILT}_MAG_ERR_2", 
                    filters=["g", "r", "i"], xlim=(28, 18), ylim_mag=(28, 18), )

## Julen

In [ ]:
cat_joined = phot_utils.outer_join_xmatch(cat, cat_julen, ra1="ra", dec1="dec", ra2="ra", dec2="dec")


In [ ]:
plot_mag_comparison(cat_joined, mag_ref_fmt="{FILT}_MAG_1", mag_cmp_fmt="{FILT}_MAG_2", 
                    mag_ref_err_fmt="{FILT}_MAG_ERR_1", mag_cmp_err_fmt="{FILT}_MAG_ERR_2", 
                    xlim=(28, 18), ylim_mag=(28, 18), )

## Photometry ssystems

In [ ]:
cat_forced = ana_utils.read_catalogue("yasone1", filter_bad=False, catname="allcolours_forced", shiftname="allcolours_panstarrs_shift")
cat = ana_utils.read_catalogue("yasone1", filter_bad=False, catname="allcolours", shiftname="allcolours_panstarrs_shift")

cat_small_bkg = ana_utils.read_catalogue("yasone1", filter_bad=False, catname="allcolours_forced_small_bkg", shiftname="allcolours_panstarrs_shift")
cat_large_bkg = ana_utils.read_catalogue("yasone1", filter_bad=False, catname="allcolours_forced_large_bkg", shiftname="allcolours_panstarrs_shift")
cat_small_bkg_filt = ana_utils.read_catalogue("yasone1", filter_bad=False, catname="allcolours_forced_small_bkg_filt", shiftname="allcolours_panstarrs_shift")
cat_large_bkg_filt = ana_utils.read_catalogue("yasone1", filter_bad=False, catname="allcolours_forced_large_bkg_filt", shiftname="allcolours_panstarrs_shift")


In [ ]:
for tab in [cat_forced, cat_small_bkg, cat_small_bkg_filt, cat_large_bkg, cat_large_bkg_filt]:
    filter_utils.calibrate_mag_col(tab, "MED_MAG_APER_3")

In [ ]:
for cat2 in [cat_small_bkg, cat_small_bkg_filt, cat_large_bkg, cat_large_bkg_filt]:
    fig, axs = plt.subplots(1, 3, figsize=(6, 2), sharex=True, sharey=True)

    cmax = 10_000
    plt.sca(axs[0])
    plt.scatter(cat_forced["ra"], cat_forced["dec"], c=cat["G_aperture_sum_3_median"] - cat2["G_aperture_sum_3_median"], vmin=-cmax, vmax=cmax, cmap="RdBu", s=1)

    plt.sca(axs[1])
    plt.scatter(cat["ra"], cat["dec"], c=cat["R_aperture_sum_3_median"] - cat2["R_aperture_sum_3_median"], vmin=-cmax, vmax=cmax, cmap="RdBu", s=1)

    plt.sca(axs[2])
    plt.scatter(cat["ra"], cat["dec"], c=cat["I_aperture_sum_3_median"] - cat2["I_aperture_sum_3_median"], vmin=-cmax, vmax=cmax, cmap="RdBu", s=1)
    plt.colorbar(label="flux difference")

In [ ]:
np.ma.median(cat_joined["R_MED_MAG_APER_3"])

In [ ]:
np.ma.median( cat_joined["R_MAG_1"])

In [ ]:
import plot_utils

In [ ]:
# plt.scatter(cat_joined["R_MAG_1"], cat_joined["R_MED_MAG_APER_3_ERR"], s=1)
# plt.scatter(cat_joined["R_MAG_1"], cat_joined["R_MAG_ERR_1"], s=1)

magerr = np.sqrt(cat_joined["G__MAG_APER_5"]**2 + cat_joined["G_MAG_ERR_1"]**2)
plt.scatter(cat_joined["G_MAG_1"], magerr, s=1, lw=0, alpha=0.1)
plot_utils.plot_sliding_window_mag_error(plt.gca(), cat_joined["G_MAG_1"], magerr)

plt.ylim(0, 1)

In [ ]:
cat_joined = phot_utils.outer_join_xmatch(cat, cat_forced, ra1="ra", dec1="dec", ra2="ra", dec2="dec")

plot_mag_comparison(cat_joined, mag_ref_fmt="{filt}_MAG_1", mag_cmp_fmt="{filt}_MED_MAG_APER_3", 
                    mag_ref_err_fmt="{filt}_MAG_ERR_1", mag_cmp_err_fmt="{filt}_MED_MAG_APER_3_ERR", 

                    ref_mag_upper=True, cmp_mag_upper=True, filters=["g", "r", "i"], xlim=(28, 18), ylim_mag=(28, 18))


In [ ]:
for cat2 in [cat_small_bkg, cat_small_bkg_filt, cat_large_bkg, cat_large_bkg_filt]:
    cat_joined = phot_utils.outer_join_xmatch(cat_forced, cat2, ra1="ra", dec1="dec", ra2="ra", dec2="dec")
    
    plot_mag_comparison(cat_joined, mag_ref_fmt="{filt}_MED_MAG_APER_3_1", mag_cmp_fmt="{filt}_MED_MAG_APER_3_2", 
                        mag_ref_err_fmt="{filt}_MED_MAG_APER_3_ERR_1", mag_cmp_err_fmt="{filt}_MED_MAG_APER_3_ERR_2", 
    
                        ref_mag_upper=True, cmp_mag_upper=True, filters=["g", "r", "i"], xlim=(28, 18), ylim_mag=(28, 18))
    plt.show()

# Yasone 2

In [ ]:
cat = ana_utils.read_catalogue("yasone2", filter_bad=False, catname="allcolours", shiftname="allcolours_panstarrs_shift")


In [ ]:
cat_lss = Table.read("../survey_data/yasone2_lss10.csv")

In [ ]:
cat_joined = phot_utils.outer_join_xmatch(cat, cat_lss, ra1="R_ALPHA_J2000", dec1="R_DELTA_J2000", ra2="ra", dec2="dec")

In [ ]:
np.sum(np.isfinite(cat_lss["mag_g"])), np.sum(np.isfinite(cat_lss["mag_r"])), np.sum(np.isfinite(cat_lss["mag_i"]))

In [ ]:
plot_mag_comparison(cat_joined, mag_ref_fmt="{filt}_MAG", mag_cmp_fmt="mag_{filt}", ref_mag_upper=True, cmp_mag_upper=False, filters=["g", "i"])

# Yasone 3

In [ ]:
cat = ana_utils.read_catalogue("yasone3", filter_bad=False, catname="allcolours", shiftname="allcolours_panstarrs_shift")


In [ ]:
cat_lss = Table.read("../survey_data/yasone3_lss10.csv")

In [ ]:
cat_joined = phot_utils.outer_join_xmatch(cat, cat_lss, ra1="ra", dec1="dec", ra2="ra", dec2="dec")

In [ ]:
np.sum(np.isfinite(cat_lss["mag_g"])), np.sum(np.isfinite(cat_lss["mag_r"])), np.sum(np.isfinite(cat_lss["mag_i"]))

In [ ]:
plot_mag_comparison(cat_joined, mag_ref_fmt="{FILT}_MAG", mag_cmp_fmt="mag_{filt}", filters=["g", "i"])
